# درس ۰۴ - الگوی طراحی استفاده از ابزار

در این درس الگوی طراحی **استفاده از ابزار** برای عوامل هوش مصنوعی با استفاده از چارچوب Microsoft Agent (پایتون) را خواهید آموخت. موارد زیر پوشش داده می‌شوند:

- تعریف ابزارهای تابع با دکوریتور `@tool` و پارامترهای تایپ شده
- ارائه شمای ابزار تا مدل بداند هر ابزار چه کاری انجام می‌دهد
- کنترل اجرای ابزار با `approval_mode`
- بازگرداندن **خروجی ساختاریافته** از طریق مدل‌های Pydantic و `response_format`

سناریو یک **عامل رزرو سفر** است که می‌تواند مقاصد را جستجو کند، در دسترس بودن را بررسی کند و اطلاعات پرواز را بازیابی کند.


## تنظیمات


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تعریف ابزارها با دکوراتور @tool

دکوراتور `@tool` یک تابع ساده پایتون را به ابزاری تبدیل می‌کند که یک عامل می‌تواند آن را فراخوانی کند.
نکات کلیدی:

- **docstring** به توصیف ابزاری تبدیل می‌شود که مدل می‌بیند.
- **توضیحات نوع** (شامل `Annotated` با شرح‌ها) طرح‌واره ابزار را تعریف می‌کنند.
- `approval_mode` کنترل می‌کند که آیا کاربر باید هر فراخوانی را قبل از اجرا تأیید کند یا خیر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایجاد یک عامل با ابزارهای متعدد

تمام سه ابزار را به کلاینت بدهید تا مدل بتواند هر کدام را که برای پاسخ به سؤال کاربر نیاز دارد، فراخوانی کند.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## خروجی ساختارمند با ابزارها

با تنظیم `response_format` به یک مدل Pydantic، عامل مجبور می‌شود که یک شیء JSON با نوع‌بندی دقیق بازگرداند به جای متن آزاد. این زمانی مفید است که کد پایین‌دستی نیاز داشته باشد نتیجه را به‌صورت برنامه‌نویسی شده مصرف کند.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## الگوهای تایید ابزار

پارامتر `approval_mode` در `@tool` کنترل می‌کند که آیا فراخوانی ابزار قبل از اجرا به تایید انسان نیاز دارد یا خیر:

| حالت | رفتار |
|---|---|
| `"never_require"` | ابزار به صورت خودکار اجرا می‌شود — نیازی به تایید کاربر نیست. |
| `"always_require"` | هر فراخوانی باید قبل از اجرا توسط کاربر تایید شود. |

برای ابزارهایی که اثرات جانبی دارند (مثلاً رزرو پرواز، شارژ کارت اعتباری) از `"always_require"` استفاده کنید تا یک انسان در چرخه باقی بماند.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصه

در این درس یاد گرفتید چگونه:

1. با استفاده از دکوراتور `@tool` ابزارها را با پارامترهای تایپ شده و docstringهایی که به عنوان طرح ابزار عمل می‌کنند، تعریف کنید.
2. چندین ابزار را با هم ترکیب کنید تا عامل بتواند آنها را به ترتیب فراخوانی کند و به پرسش‌های پیچیده پاسخ دهد.
3. خروجی ساختاریافته‌ای را با ارسال یک مدل Pydantic به عنوان `response_format` بازگردانید.
4. با استفاده از `approval_mode` کنترل تأیید ابزار را داشته باشید تا یک انسان در عملیات حساس دخیل باشد.

این الگوها پایه‌ای برای ساخت عامل‌های قابل اعتماد و آماده تولید هستند که می‌توانند به‌طور ایمن با سیستم‌های خارجی تعامل داشته باشند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح مسئولیت**:  
این سند با استفاده از سرویس ترجمه خودکار هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما تلاش می‌کنیم دقت را حفظ کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادقیق‌گی‌هایی باشند. سند اصلی به زبان بومی خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سوءتفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
